In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
from io import StringIO
from hallmark.eht_datatree import build_repo
from hallmark import Repo
from collections import defaultdict

pd.set_option("display.width", 300)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

DATASETS = ["EHTC_FirstSgrAResults_May2022","EHTC_CenA2017_Jul2021",
           "EHTC_First3C279Results_May2020","EHTC_FirstM87Results_Apr2019",
           "EHTC_FirstSgrAPol_Mar2024","EHTC_M87-2018_Mar2024",
           "EHTC_M87mwl2017_Apr2021","EHTC_M87mwl2018_Dec2024",
           "EHTC_M87pol2017_Nov2023","EHTC_metadata2018_Dec2023",
           "EHTC_MonitoringM87_Sep2020","EHTC_SgrAmwl2017_May2022"]
idx = 0
FMT = "ER6_SGRA_2017_{scan}_{band}_{pipeline}_netcal-LMTcal-{method}_StokesI"
ROOT = Path(f"~/{DATASETS[idx]}").expanduser()
REPO_PATH = Path(f"~/{DATASETS[idx]}_hallmark").expanduser()

In [ ]:
repo = build_repo(root=ROOT, repo_path=REPO_PATH, dataset_name=DATASETS[idx],
                   fmt=FMT, overwrite=True)
print("Repo built at:", repo.dothm.path)

branches = repo.branches()
print(f"Current branch: {branches['current']}")
print(f"\nTotal branches: {len(branches['names'])}")

print("\nmain/")
print("  meta.yaml:")
for f in repo.state.meta.get("files", []):
    print(f"    {Path(f).name}")

fmt_prefixes = sorted({
    name.split("/")[0]
    for name in branches["names"]
    if name != "main"
})

for fmt in fmt_prefixes:
    first_stem = next(
        name for name in branches["names"]
        if name.startswith(fmt + "/")
    )
    repo.dothm.git.checkout(first_stem)
    repo.state = repo.dothm.load()
    real_fmt = repo.state.config["data"][0]["fmt"]
    stem_count = sum(1 for n in branches["names"] if n.startswith(fmt + "/"))
    print(f"\n{real_fmt}/  \n  ({stem_count} stems)")

repo.dothm.git.checkout("main")
repo.state = repo.dothm.load()



In [ ]:
# get unique fmt prefixes
fmt_prefixes = sorted({
    name.split("/")[0]
    for name in branches["names"]
    if name != "main"
})

print()
for fmt_prefix in fmt_prefixes:
    # get real fmt string from first stem's config
    first_stem = next(
        b for b in sorted(branches["names"])
        if b.startswith(fmt_prefix + "/")
    )
    real_fmt = repo.dothm.git.show(f"{first_stem}:config.yml")
    import yaml
    config = yaml.safe_load(real_fmt)
    fmt_str = config["data"][0]["fmt"]
    
    # get stems for this fmt
    fmt_branches = sorted(b for b in branches["names"] 
                          if b.startswith(fmt_prefix + "/"))
    
    # get param cols from first stem
    first_data = pd.read_csv(
        StringIO(repo.dothm.git.show(f"{first_stem}:data.tsv")),
        sep="\t", dtype=str
    )
    param_cols = [c for c in first_data.columns if c != "sha1"]
    
    header = f"  {'STEM':<35} {'FILES':<8} " + "  ".join(f"{col.upper():<10}" 
                                                         for col in param_cols)
    print(f"fmt: {fmt_str}")
    print(header)
    print("  " + "=" * (len(header) - 2))
    
    for branch in fmt_branches:
        stem = branch.split("/")[-1]
        data = pd.read_csv(
            StringIO(repo.dothm.git.show(f"{branch}:data.tsv")),
            sep="\t", dtype=str
        )
        row = data.iloc[0]
        params = "  ".join(f"{str(row.get(col, 'NaN')):<10}" for col in param_cols)
        print(f"  {stem:<35} {len(data):<8} {params}")
    
    print(f"  Total: {len(fmt_branches)} stems\n")

In [ ]:
target_branch = next(b for b in sorted(branches["names"]) if b != "main")

repo.dothm.git.checkout(target_branch)
repo.state = repo.dothm.load()

print(f"Branch: {target_branch.split('/')[-1]}")
print(f"\nConfig:")
print(f"  fmt: {repo.state.config['data'][0]['fmt']}")
print(f"  encoding: {repo.state.config['data'][0]['encoding']}")
print(f"  remote: {repo.state.config.get('remote')}")
print(f"\nData:")
print(repo.state.data)

# return to main
repo.dothm.git.checkout("main")
repo.state = repo.dothm.load()

In [33]:
from hallmark.repo_manifest import fmt_fields
print(fmt_fields("SR1_M87_{year}_{day}_{band}_hops_netcal_StokesI.{ext}"))

['year', 'day', 'band', 'ext']
